# Phase 1 Task B — ConvNeXt-Tiny × 3 Seeds

**Model:** ConvNeXt-Tiny
**Seeds:** 42, 123, 456
**Batch:** 32
**Output:** wandb `woodbench-taskB`

Chạy notebook này trên 1 GPU (vast.ai).

In [ ]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break
import sys, os
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

In [ ]:
# Load .env for API keys
from dotenv import load_dotenv
dotenv_path = ROOT / ".env"
load_dotenv(dotenv_path)
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops", "python-dotenv"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
os.environ.setdefault("HF_TOKEN", os.getenv("HF_TOKEN", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")
print(f"[HF] token: {'set' if os.getenv('HF_TOKEN') else 'MISSING'}")

In [ ]:
# Download data from HuggingFace
HF_REPO = "ny0g507h3p/wood-task-B"
DATA_ROOT = ROOT / "data" / "task-B"

if not (DATA_ROOT / "train").exists():
    print(f"[HF] Downloading dataset to {DATA_ROOT}...")
    from huggingface_hub import login, snapshot_download
    token = os.getenv("HF_TOKEN", "")
    if token:
        login(token=token)
        print("[HF] Logged in")
    else:
        print("[HF] No token — trying public access")
    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        local_dir=str(DATA_ROOT),
        ignore_patterns=["*.md", "README*"],
    )
    print("[HF] Download complete")
else:
    print(f"[Data] Found at {DATA_ROOT}")

# Verify
train_dir = DATA_ROOT / "train"
if not train_dir.exists():
    raise FileNotFoundError(f"[ERROR] train/ not found in {DATA_ROOT}. Dataset download failed.")
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"[Data] {len(classes)} classes: {classes[:3]}...")
print(f"[Data] Train samples: {sum(1 for _ in train_dir.rglob('*.jpg')) + sum(1 for _ in train_dir.rglob('*.png'))}")

In [ ]:
from src.data import get_dataloaders
# num_workers=0 to avoid multiprocessing issues on vast.ai
tl, vl, testl = get_dataloaders(str(DATA_ROOT), batch_size=32, num_workers=0, task="taskB")
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

In [ ]:
import wandb, torch, numpy as np
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

In [ ]:
def train_one(backbone, seed, data_root, batch_size=32, epochs=50):
    set_seed(seed)
    output_dir = ROOT / "experiments" / "taskB" / backbone / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    # num_workers=0 for papermill compatibility
    train_loader, val_loader, _ = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=0, task="taskB")
    num_classes = len(train_loader.dataset.classes)
    print(f"[{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}")

    run = wandb.init(project="woodbench-taskB", name=f"{backbone}_s{seed}",
        config={"backbone": backbone, "seed": seed, "epochs": epochs,
                "batch_size": batch_size, "num_classes": num_classes,
                "lr_head": 1e-4, "lr_backbone": 1e-5,
                "label_smoothing": 0.1, "scheduler": "cosine", "amp": True},
        dir=str(output_dir))

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True)
    result = engine.fit(epochs=epochs, output_dir=output_dir,
                       name=f"{backbone}_seed{seed}", save_best_only=False)
    run.finish()
    print(f"[{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    return result

In [ ]:
SEEDS = [42, 123, 456]
results = {}

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"ConvNeXt-Tiny | seed={seed} ({SEEDS.index(seed)+1}/{len(SEEDS)})")
    print(f"{'='*60}")
    results[seed] = train_one("convnext_tiny", seed, str(DATA_ROOT), batch_size=32, epochs=50)

print("\n" + "="*60)
print("ConvNeXt-Tiny — Summary")
for seed, res in results.items():
    print(f"  seed={seed}: {res['best_val_acc']:.2f}%")
accs = [r['best_val_acc'] for r in results.values()]
print(f"  Mean: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")